In [0]:
import time

import pyspark.sql.functions as F

CATALOG = 'car_workshop'
LAB = f'{CATALOG}.lab'

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {LAB}')
spark.sql(f'CREATE VOLUME IF NOT EXISTS {LAB}.files')
LAB_DIR = f'/Volumes/{CATALOG}/lab/files'


def timed(label, fn):
    t0 = time.time()
    result = fn()
    print(f'{label}: {time.time() - t0:.1f}s')
    return result


dbutils.widgets.dropdown("Is cluster mode?", "False", ["True", "False"])
dbutils.widgets.text("Rows multiplier", "10")
is_cluster_mode = dbutils.widgets.get("Is cluster mode?")
MULT = int(dbutils.widgets.get("Rows multiplier"))

print(f'lab schema: {LAB}, lab volume: {LAB_DIR}, row multiplier: {MULT}')


%md
# Spark UI diagnosis lab

**The interview question this notebook makes visible:**

> *"Your autoscaling cluster keeps hitting max workers, but jobs aren't faster.
> What's happening?"*

The senior answer is a **Spark UI walk**, not a config change. Each demo below
reproduces one reason why "more workers" buys nothing, and tells you exactly
which UI screen proves it:

| # | why extra workers don't help            | what you'll see in the UI                                   |
|---|-----------------------------------------|-------------------------------------------------------------|
| 1 | parallelism capped by partition count   | stage with 4 tasks on a cluster with 8+ cores               |
| 2 | skew: one straggler pins the wall clock | task Duration: **Max >> Median** in Summary Metrics         |
| 3 | work isn't on the cluster at all        | **gap** in the jobs timeline / a 1-task write stage         |
| 4 | tiny tasks: overhead > compute          | hundreds of tasks with millisecond durations                |

**How to open the Spark UI on Databricks (classic cluster):**
- fastest: under an executed cell expand **"Spark Jobs"** → click **View** next to a job/stage — it deep-links into the UI;
- or: **Compute → your cluster → "Spark UI" tab** → Jobs / Stages / Executors / SQL-DataFrame tabs.

**Serverless has no classic Spark UI.** You get the **query profile** instead
(cell dropdown → *"View query profile"*, or Query History). It shows per-operator
task counts, rows and memory — enough to spot demos 1–2, but the executor/timeline
views need a classic cluster. That's why the juicy cells are gated on the
`Is cluster mode?` widget, same convention as `spark_deep_dive.ipynb`.

Run order: setup → data build → demos in any order. `Rows multiplier` scales the
lab table (10 ≈ enough to make stages take seconds, not milliseconds — raise it
if your cluster is beefy and everything finishes too fast to see).


In [0]:
# build the lab tables once: fact_sales_items replicated MULT times.
# grp_key = synthetic high-cardinality aggregation key (~2M groups) so groupBy
# produces a REAL shuffle instead of a trivial one
items = spark.table(f'{CATALOG}.fact.fact_sales_items')

(items
 .crossJoin(spark.range(MULT).withColumnRenamed('id', 'replica'))
 .withColumn('grp_key', F.pmod(F.hash('sales_item_id', 'replica'), F.lit(2_000_000)))
 .write.mode('overwrite').saveAsTable(f'{LAB}.ui_sales_big'))

# skewed twin: 85% of rows -> one hot product (straggler material for demo 2,
# same trick as data_skewness.ipynb)
hot_product = items.groupBy('product_id').count().orderBy(F.desc('count')).first()['product_id']

(spark.table(f'{LAB}.ui_sales_big')
 .withColumn('product_id',
             F.when(F.rand(seed=42) < 0.85, F.lit(hot_product)).otherwise(F.col('product_id')))
 .write.mode('overwrite').saveAsTable(f'{LAB}.ui_sales_skewed'))

for t in ['ui_sales_big', 'ui_sales_skewed']:
    print(t, f"{spark.table(f'{LAB}.{t}').count():,} rows")


In [0]:
# STEP 0 of the interview answer: before judging utilization you must know the
# denominator - how many task slots the cluster actually has
if is_cluster_mode == 'True':
    print('total cores available to Spark:', sc.defaultParallelism)
    print('-> a stage with FEWER tasks than this number CANNOT keep the cluster busy,')
    print('   and adding workers only raises the denominator, not the numerator')
else:
    print('serverless: core count is elastic and hidden - reason from the query')
    print('profile task counts instead (classic cluster recommended for this lab)')


%md
## Demo 1 — parallelism is capped by partition count, not core count

This is the heart of the interview answer: **a shuffle stage runs exactly
`spark.sql.shuffle.partitions` tasks**. If that's 4 and you autoscale from 8 to
80 cores, 76 cores are idle — and the autoscaler may still report "busy" because
those 4 fat tasks queue follow-up work.

**Run the next cell, then open the Spark UI and check:**
1. **Jobs tab** → the job from run A → its second stage shows **"4 tasks"**. That number is the hard ceiling on parallelism for that stage.
2. Stage detail → **Event Timeline**: only 4 lanes ever busy, no matter the cluster size.
3. **Executors tab**: compare *Task Time* across executors — most did (almost) nothing during run A.
4. Repeat for run B (200 tasks): every core has work, wall clock drops.

Interview phrasing: *"first I check whether the stage even has enough tasks to
occupy the cores we already pay for — shuffle parallelism is bounded by the
partition count, so past that bound new workers are pure cost."*


In [0]:
if is_cluster_mode == 'False':
    print("Skipping - conf toggles are not on the serverless allowlist; see next cell")
else:
    # CLASSIC CLUSTER ONLY - AQE off so the raw partition count is what actually runs
    spark.conf.set('spark.sql.adaptive.enabled', 'false')

    big = spark.table(f'{LAB}.ui_sales_big')

    def capped_agg():
        return (big.groupBy('grp_key')
                   .agg(F.sum('value_gross').alias('sum_gross'),
                        F.avg('unit_price_net').alias('avg_price'))
                   # sha2 AFTER the shuffle = CPU work trapped inside the capped stage,
                   # so the 4-task ceiling really hurts and is easy to see on the timeline
                   .withColumn('fp', F.sha2(F.concat_ws('|', 'grp_key', 'sum_gross'), 256))
                   .agg(F.count('*'), F.max('fp'))
                   .collect())

    spark.conf.set('spark.sql.shuffle.partitions', '4')
    timed('A: shuffle.partitions=4   (cluster mostly idle)', capped_agg)

    spark.conf.set('spark.sql.shuffle.partitions', '200')
    timed('B: shuffle.partitions=200 (same data, same cluster)', capped_agg)

    # reset to Databricks defaults
    spark.conf.set('spark.sql.adaptive.enabled', 'true')
    spark.conf.set('spark.sql.shuffle.partitions', 'auto')


In [0]:
# SERVERLESS variant - you cannot pin shuffle.partitions, the engine picks it.
# But you can still OBSERVE the chosen parallelism (no RDD API needed):
def n_partitions(df):
    """Count distinct physical partitions the rows actually landed in."""
    return (df.withColumn('_pid', F.spark_partition_id())
              .select(F.countDistinct('_pid')).first()[0])


big_sl = spark.table(f'{LAB}.ui_sales_big')
print('engine-chosen partitions after groupBy shuffle:',
      n_partitions(big_sl.groupBy('grp_key').agg(F.sum('value_gross'))))
# then open the query profile: each shuffle/exchange node reports its task count -
# that number (not the worker count) is the parallelism ceiling of the stage


%md
## Demo 2 — skew: one straggler pins the runtime, new workers watch

85% of `ui_sales_skewed` points at one hot `product_id`. A sort-merge join
partitions by that key, so **one task gets 85% of the data**. The stage — and the
whole job — finishes when *that* task finishes. Autoscaling sees pending work,
scales to max, and the new workers idle while one core grinds.

**Run the next cell, then in the Spark UI (run A):**
1. Longest stage → **Summary Metrics** table: *Duration* row — **Max is many × the Median**. That gap **is** the skew; a healthy stage has Max ≈ 75th percentile.
2. Same table, *Shuffle Read Size / Records*: one task read a giant slice.
3. **Tasks** list → sort by Duration descending → meet your straggler.
4. Stage **Event Timeline**: a lonely long bar while other lanes are empty — the exact picture of "max workers, zero speedup".
5. Run B (AQE skew rescue): **SQL / DataFrame tab** → the query → the join node shows `skew=true` / `AQEShuffleRead` splitting the hot partition; Max vs Median tightens.

The real-world fixes are AQE skew join, broadcasting the small side, or salting —
`data_skewness.ipynb` drills those. Here the point is **recognising skew in the UI
before spending money on workers**.


In [0]:
if is_cluster_mode == 'False':
    print("Skipping - forcing SMJ / toggling AQE not possible on serverless; see next cell")
else:
    # CLASSIC CLUSTER ONLY
    skewed = spark.table(f'{LAB}.ui_sales_skewed')
    products = spark.table(f'{CATALOG}.dim.dim_products')

    # run A: forbid broadcast -> real sort-merge join; AQE off -> no skew rescue
    spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
    spark.conf.set('spark.sql.adaptive.enabled', 'false')
    timed('A: skewed SMJ, no rescue (find the straggler)',
          lambda: skewed.join(products, 'product_id')
                        .agg(F.sum('value_gross')).collect())

    # run B: AQE skew-join ON, thresholds lowered so lab-sized data triggers the split
    # (defaults are 256MB / factor 5 - tuned for prod-sized partitions)
    spark.conf.set('spark.sql.adaptive.enabled', 'true')
    spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')
    spark.conf.set('spark.sql.adaptive.skewJoin.skewedPartitionFactor', '2')
    spark.conf.set('spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes', '8MB')
    timed('B: same join, AQE skew rescue ON',
          lambda: skewed.join(products, 'product_id')
                        .agg(F.sum('value_gross')).collect())

    # reset to defaults
    spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '10MB')
    spark.conf.set('spark.sql.adaptive.skewJoin.skewedPartitionFactor', '5')
    spark.conf.set('spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes', '256MB')


In [0]:
# SERVERLESS variant - the engine will likely broadcast the dim and/or auto-rescue
# the skew, so you diagnose the DATA, not the stage: measure the imbalance that
# WOULD become a straggler on a fixed plan
skewed_sl = spark.table(f'{LAB}.ui_sales_skewed')

dist = skewed_sl.groupBy('product_id').agg(F.count('*').alias('cnt'))
stats = dist.select(
    F.max('cnt').alias('max_cnt'),
    F.expr('percentile_approx(cnt, 0.5)').alias('median_cnt'),
).first()
print(f"hottest key: {stats['max_cnt']:,} rows | median: {stats['median_cnt']:,} "
      f"| skew ratio: {stats['max_cnt'] / stats['median_cnt']:.0f}x")
# in the query profile the join node still reports per-task row max/avg -
# a huge max/avg ratio there is the same straggler signal


%md
## Demo 3 — the cluster isn't doing the work at all

Two classics that autoscaling can never fix, because **no amount of workers
executes code that doesn't run on workers**:

- **A. driver-bound work**: `toPandas()` + a Python loop. The cluster finishes the
  collect, then sits idle while one driver core loops.
- **B. the non-parallel tail**: `coalesce(1)` write — the whole output funnels
  through **one task**.

**In the Spark UI:**
1. **Jobs tab → Event Timeline** (top of the page): after the collect job there's a **blank gap** with no jobs running — that gap *is* your pandas loop. Cluster at max workers, utilization ~zero.
2. For the `coalesce(1)` write: the write job's final stage shows **1 task**, and the Executors tab shows exactly one executor accumulating task time.
3. Compare with the parallel write: same data, final stage has many tasks.

Interview phrasing: *"gaps in the jobs timeline mean the bottleneck is outside
the cluster — driver code, notebook logic, an external API — and scaling workers
is spending on the part that was already fast."*


In [0]:
big = spark.table(f'{LAB}.ui_sales_big')

# --- A) driver-bound: collect + pure-Python loop (works on both modes) ---
sample = big.select('value_gross').limit(1_000_000)
pdf = timed('collect 1M rows to the driver', lambda: sample.toPandas())
timed('python for-loop over 1M rows (DRIVER only - watch the timeline gap)',
      lambda: sum(v for v in pdf['value_gross']))
timed('same sum as a Spark agg (distributed, no gap)',
      lambda: sample.agg(F.sum('value_gross')).collect())

# --- B) non-parallel tail: coalesce(1) vs normal write ---
timed('coalesce(1) write - ONE task regardless of worker count',
      lambda: big.coalesce(1).write.mode('overwrite')
                 .parquet(f'{LAB_DIR}/ui_single_file'))
timed('normal parallel write',
      lambda: big.write.mode('overwrite')
                 .parquet(f'{LAB_DIR}/ui_parallel'))


%md
## Demo 4 — tiny tasks: scheduling overhead beats compute

The opposite failure: **too much parallelism**. Hundreds of millisecond-sized
tasks mean the cluster spends its time scheduling, serializing and fetching —
not computing. Extra workers *do* absorb more tiny tasks, so the job gets
*slightly* faster, which is exactly the trap: cost grows linearly, speedup
doesn't (Amdahl says hi).

**In the Spark UI (run A):**
1. Scan stage → task count in the hundreds, **Median duration in milliseconds**.
2. **Summary Metrics** → *Scheduler Delay* + *Task Deserialization Time* are a visible share of each task — pure overhead, paid per task.
3. Run B: a handful of right-sized tasks, overhead share collapses.

Real-world root cause is usually **small files** (streaming sinks, per-partition
writes) — the fix is compaction / `OPTIMIZE`, not workers.


In [0]:
# build the two layouts once: same rows, wildly different file counts
big = spark.table(f'{LAB}.ui_sales_big')
big.repartition(600).write.mode('overwrite').parquet(f'{LAB_DIR}/ui_many_files')
big.repartition(8).write.mode('overwrite').parquet(f'{LAB_DIR}/ui_few_files')

if is_cluster_mode == 'False':
    # serverless packs small files into tasks on its own - just compare and read
    # the query profile ("files read" / task counts per scan node)
    timed('A: scan 600 tiny files', lambda: spark.read.parquet(f'{LAB_DIR}/ui_many_files')
          .agg(F.sum('value_gross')).collect())
    timed('B: scan 8 right-sized files', lambda: spark.read.parquet(f'{LAB_DIR}/ui_few_files')
          .agg(F.sum('value_gross')).collect())
else:
    # CLASSIC CLUSTER ONLY: by default Spark PACKS small files into ~128MB tasks
    # (maxPartitionBytes + openCostInBytes), hiding the problem at this data size.
    # Shrink the pack size so the scan really launches ~1 task per file:
    spark.conf.set('spark.sql.files.maxPartitionBytes', '1MB')
    timed('A: scan 600 tiny files (1 task/file - overhead city)',
          lambda: spark.read.parquet(f'{LAB_DIR}/ui_many_files')
                       .agg(F.sum('value_gross')).collect())
    spark.conf.set('spark.sql.files.maxPartitionBytes', '128MB')
    timed('B: scan 8 right-sized files',
          lambda: spark.read.parquet(f'{LAB_DIR}/ui_few_files')
                       .agg(F.sum('value_gross')).collect())


%md
## The senior answer, as a repeatable UI walk

When someone says *"we hit max workers but jobs aren't faster"*, walk the UI in
this order — each step maps to a demo above:

1. **Executors tab** — is task time spread across executors, or concentrated?
   Idle executors at max workers = the money is already wasted; now find why.
2. **Jobs timeline** — gaps between jobs? The bottleneck is the **driver** or
   external calls (demo 3). No cluster size fixes that.
3. **Stage task count vs. total cores** — fewer tasks than cores (demo 1)?
   Parallelism is capped by **partition count**; fix partitioning, not workers.
4. **Stage Summary Metrics: Max vs Median duration** — huge gap = **skew**
   (demo 2). One straggler pins the wall clock; AQE skew join / broadcast / salt.
5. **Median task duration** — milliseconds (demo 4)? Overhead-bound tiny tasks;
   compact files. Seconds-to-a-minute with high shuffle read/write and **spill**?
   The stage is disk/network-bound — better partition sizing or fewer, bigger
   nodes beat more nodes.

**When DO extra workers help?** Many independent, healthy-sized (~100–200MB
input) tasks, CPU-bound, no straggler — i.e. the Stages tab shows a deep queue of
uniform tasks. That's the only picture where raising max workers buys time.

**And the autoscaling part of the answer:** autoscaling earns its keep on bursty,
variable load. For a steady daily batch like this repo's `daily.ipynb` →
`autoloader.ipynb` → silver cycle, a right-sized **fixed** cluster (or a narrow
min/max band) is cheaper and more predictable than a wide band that thrashes up
and down every run.


In [0]:
# cleanup when done - only touches this notebook's ui_* objects in the lab schema
for t in ['ui_sales_big', 'ui_sales_skewed']:
    spark.sql(f'DROP TABLE IF EXISTS {LAB}.{t}')
for d in ['ui_single_file', 'ui_parallel', 'ui_many_files', 'ui_few_files']:
    dbutils.fs.rm(f'{LAB_DIR}/{d}', recurse=True)
print('lab ui_* objects removed')
